# 性能改善：正しさ・時間・メモリを測る

対応する本文は [performance.tex](../chapters/performance.tex) である。コードセルを上から順にShift+Enterで実行する。各問の説明にある値や図と、セルの下に表示される結果を比べる。ファイルを作る問では、その直前に保存先を用意する。

## 問1：二乗和の結果を照合して測定する

浮動小数点の加算順序による丸めの差があるため、相対許容差を明示する。入力サイズ、Python と NumPy の版、反復回数を記録する。5回の中央値を使い、OS やキャッシュによる変動も見る。

この後のセルにある `assert 条件` は、結果を自動で確認する命令である。条件が違っていればエラーで止まり、条件を満たせば次へ進む。何も表示されなくても、エラーが出なければその確認は通っている。画面の数値や図も、説明に示した値と比べる。

In [1]:
import statistics
import time

import numpy as np
import pandas as pd

values = np.arange(1_000_000, dtype=np.float64)
def python_sum_squares(values):
    total = 0.0
    for value in values:
        total += value * value
    return total

def numpy_sum_squares(values):
    return np.sum(values * values)

small = np.array([1.0, 2.0, 3.0])
print(python_sum_squares(small), numpy_sum_squares(small))

14.0 14.0


### 14になることを確かめてから時間を測る

二乗和は、各値を二乗して足したものである。1、2、3なら `1 + 4 + 9 = 14` なので、上の表示は両方14.0になる。

次の `measure()` は、同じ関数を5回呼び、各回の経過時間をリストへ入れる。`time.perf_counter()` で前後の時刻を取り、その差を秒で求める。関数を引数として受け取るため、Python版とNumPy版へ同じ測定方法を使える。

In [2]:
def measure(function, *args, repeats=5):
    samples = []
    for _ in range(repeats):
        start = time.perf_counter()
        function(*args)
        samples.append(time.perf_counter() - start)
    return samples



次は100万個の値で、答えが一致することを確かめてから測定する。`assert_allclose()` の `rtol` は割合での許容差、`atol` は絶対値での許容差である。表示する「中央値」は5回の時間を小さい順に並べたときの中央の値である。処理時間は計算機や同時に動く処理によって変わる。

In [3]:
np.testing.assert_allclose(python_sum_squares(values), numpy_sum_squares(values), rtol=1e-10, atol=0)
measurements = {name: measure(function, values) for name, function in
                [("Python", python_sum_squares), ("NumPy", numpy_sum_squares)]}
for name, samples in measurements.items():
    print(name, "各回 [s]:", samples, "中央値 [s]:", statistics.median(samples))
print("NumPy", np.__version__, "要素数", len(values), "反復数", 5)

Python 各回 [s]: [0.050501291174441576, 0.04814529186114669, 0.04870383394882083, 0.049321750178933144, 0.05242079170420766] 中央値 [s]: 0.049321750178933144
NumPy 各回 [s]: [0.0005802498199045658, 0.00025066686794161797, 0.0002409159205853939, 0.0002499171532690525, 0.00023974990472197533] 中央値 [s]: 0.0002499171532690525
NumPy 2.5.4 要素数 1000000 反復数 5


## 問2：入力サイズと時間比を比べる

重複があると途中で終了して比較数が変わるため、重複のないランダム順の入力を使う。全ペア比較の時間比はおよそ4、ソートを含む処理は `2 log(2N) / log(N)` が目安となる。小規模入力では定数項と測定誤差も効くため、比の厳密な一致は要求しない。

ここでNは入力の個数であり、slow_ratioとfast_ratioは一つ前の入力サイズでの時間に対する比である。例えば1秒から4秒へ増えれば比は4になる。最初の行には比較する前の測定がないためNaNが表示される。O(N²)などの記法は、入力が大きくなったときの処理回数の増え方を表す。実測秒数そのものではない。

### 保存先を用意する

この後のセルではファイルを作る。次のセルを実行して、`notebooks/_outputs/12_performance/` を用意する。表示された場所に結果が保存され、同名の実習用ファイルは再実行すると置き換わる。

Notebookは教材の `notebooks` フォルダー、または教材の一番上のフォルダーから実行する。`Path.cwd()` は現在のフォルダーを調べ、`mkdir()` は保存先を作る。

In [4]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "main.tex").is_file():
    raise FileNotFoundError("教材のフォルダーか notebooks フォルダーから実行してください。")
WORK = ROOT / "notebooks" / "_outputs" / "12_performance"
WORK.mkdir(parents=True, exist_ok=True)
print("Python", sys.version.split()[0])
print("保存先:", WORK.relative_to(ROOT))

Python 3.12.14
保存先: notebooks/_outputs/12_performance


In [5]:
def has_duplicate_slow(values):
    for i in range(len(values)):
        for j in range(i + 1, len(values)):
            if values[i] == values[j]:
                return True
    return False

def has_duplicate_fast(values):
    ordered = sorted(values)
    return any(ordered[i] == ordered[i+1] for i in range(len(ordered)-1))

rows = []
for n in [1000, 2000, 4000]:
    data = np.random.default_rng(42).permutation(n).tolist()
    assert not has_duplicate_slow(data) and not has_duplicate_fast(data)
    rows.append({"N": n, "slow_s": statistics.median(measure(has_duplicate_slow, data)),
                 "fast_s": statistics.median(measure(has_duplicate_fast, data))})
scaling = pd.DataFrame(rows)
scaling["slow_ratio"] = scaling.slow_s / scaling.slow_s.shift()
scaling["fast_ratio"] = scaling.fast_s / scaling.fast_s.shift()
print(scaling)
scaling.to_csv(WORK / "scaling.csv", index=False)
assert has_duplicate_slow([1, 2, 1]) and has_duplicate_fast([1, 2, 1])

      N    slow_s    fast_s  slow_ratio  fast_ratio
0  1000  0.007564  0.000050         NaN         NaN
1  2000  0.028822  0.000122    3.810252    2.410373
2  4000  0.116893  0.000292    4.055767    2.400819


## 問3：ビューとコピー、型ごとの容量を確認する

ビューは元のデータを共有する。条件抽出はコピーを作るので、その書き換えは元に届かない。`nbytes` は要素領域の容量であり、管理情報やプロセス全体のメモリとは異なる。

In [6]:
values = np.arange(1_000_000, dtype=np.float64)
view = values[100:200]
copied = values[values % 2 == 0]
view[0] = -1
copied[0] = -2
assert values[100] == -1 and values[0] == 0
assert np.shares_memory(values, view) and not np.shares_memory(values, copied)
print("values[100], values[0]:", values[100], values[0])
print("float32:", values.astype(np.float32).nbytes, "float64:", values.nbytes)
assert values.astype(np.float32).nbytes * 2 == values.nbytes

values[100], values[0]: -1.0 0.0
float32: 4000000 float64: 8000000


## 問4：逐次集計と全件読み込みを比較する

逐次集計は合計と件数だけを保持するため、集計用の記憶量は O(1) である。全件リストは O(N) となる。ここでは `tracemalloc` により、それぞれの関数の実行中に Python が確保した領域のピークを測る。ネイティブ領域やプロセス全体を測る値ではない。

In [7]:
import tracemalloc

path = WORK / "values.txt"
with path.open("w", encoding="utf-8") as f:
    for value in range(1, 100001):
        f.write(f"{value}\n")

def streaming_mean(path):
    count, total = 0, 0.0
    with Path(path).open(encoding="utf-8") as f:
        for line in f:
            if line.strip():
                total += float(line)
                count += 1
    if count == 0:
        raise ValueError("入力が空です。")
    return total / count

def loaded_mean(path):
    with Path(path).open(encoding="utf-8") as f:
        data = [float(line) for line in f if line.strip()]
    return sum(data) / len(data)

for function in [streaming_mean, loaded_mean]:
    tracemalloc.start()
    try:
        result = function(path)
        _, peak = tracemalloc.get_traced_memory()
    finally:
        tracemalloc.stop()
    assert result == 50000.5
    print(function.__name__, "平均:", result, "Python 領域のピーク [bytes]:", peak)

streaming_mean 平均: 50000.5 Python 領域のピーク [bytes]: 21973
loaded_mean 平均: 50000.5 Python 領域のピーク [bytes]: 3215330


## 問5：プロファイルから改善対象を選ぶ

`ncalls` は呼び出し回数、`tottime` はその関数自身の処理時間、`cumtime` は呼び出し先を含む累積時間である。下では `cProfile` の Python API を使い、端末の `python -m cProfile -s cumtime script.py` と同じ統計を得る。

二乗和の Python ループは各要素の処理が Python で実行されるため、改善候補となる。NumPy の内部計算や呼び出し回数が少ない処理は、測定上の寄与と既存実装の効率を見て優先度を決める。累積時間の親と子を合算すると二重計数になることにも注意する。

In [8]:
import cProfile
import io
import pstats

profile_values = np.arange(1_000_000, dtype=np.float64)
profiler = cProfile.Profile()
profiler.runcall(python_sum_squares, profile_values)
profiler.runcall(numpy_sum_squares, profile_values)
stream = io.StringIO()
stats_report = pstats.Stats(profiler, stream=stream).strip_dirs().sort_stats("cumulative")
stats_report.print_stats(3)
print(stream.getvalue())
(WORK / "profile_top3.txt").write_text(stream.getvalue(), encoding="utf-8")

         63 function calls in 0.021 seconds

   Ordered by: cumulative time
   List reduced from 29 to 3 due to restriction <3>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        2    0.000    0.000    0.020    0.010 base_events.py:1922(_run_once)
        2    0.000    0.000    0.020    0.010 selectors.py:558(select)
        1    0.017    0.017    0.017    0.017 {method 'control' of 'select.kqueue' objects}





443